# Ramsey-vs-Q-axis analysis

Analysis for `ramsey_qaxis_scan.py`, modeled on `analyze_mw_scan_threshold.ipynb`.
Each data point carries `rqs_shimY` (the Y-shim voltage = q-axis strength). The
notebook groups by B-field and, for **each** B-field, fits the Ramsey fringe
(`RamseyFit`, LSQ) and shows the 4-segment zoom (0/8/16/24 ms windows).

In [ ]:
import sys, os
sys.path.insert(0, r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository")
import numpy as np
import matplotlib.pyplot as plt
import h5py, glob, datetime
from pathlib import Path
from analysis.data_loader import load_h5, find_runs, RESULTS_DIR
from fitting.damped_cosine import RamseyFit

%matplotlib inline
%load_ext autoreload
%autoreload 2
plt.rcParams["figure.dpi"] = 120

def _scalar(v, default=0.0):
    if v is None: return float(default)
    if isinstance(v, (bytes, bytearray)):
        try: return float(v.decode())
        except Exception: return float(default)
    arr = np.asarray(v)
    if arr.ndim == 0: return float(arr)
    if arr.size == 0: return float(default)
    return float(arr.flat[-1])

def _txt(v, default="?"):
    if isinstance(v, (bytes, bytearray)): return v.decode()
    return default if v is None else str(v)

print("Setup complete. Results dir:", RESULTS_DIR)

## 1. List recent RamseyQaxisScan runs

In [ ]:
files = find_runs(experiment="RamseyQaxisScan", format="h5")
if not files:
    print("No RamseyQaxisScan HDF5 files in", RESULTS_DIR)
else:
    print(f"{'RID':>9s}  {'Mode':<12s}  {'Date':<12s}  {'Time':<6s}  Folder")
    print("-" * 56)
    for f in files[-20:]:
        rid = os.path.basename(f).split("-")[0]
        dt = datetime.datetime.fromtimestamp(os.path.getmtime(f))
        mode = ""
        try:
            with h5py.File(f, "r") as hf:
                ds = hf.get("datasets")
                if ds is not None and "rqs_scan_mode" in ds:
                    mode = _txt(ds["rqs_scan_mode"][()])
        except Exception:
            pass
        folder = str(Path(f).parent.relative_to(RESULTS_DIR))
        print(f"{rid:>9s}  {mode:<12s}  {dt.strftime('%Y-%m-%d'):<12s}  {dt.strftime('%H:%M'):<6s}  {folder}")

## 2. Select run
Set `RID` (or `None` for the latest RamseyQaxisScan run), then load and extract per-shot arrays.

In [ ]:
########################################
RID = None   # set an int RID, or None for latest
########################################
if RID is not None:
    matches = glob.glob(str(RESULTS_DIR / "**" / f"{RID:09d}-*.h5"), recursive=True)
    if not matches: raise FileNotFoundError(f"No HDF5 for RID {RID}")
    filepath = max(matches, key=os.path.getmtime)
else:
    files = find_runs(experiment="RamseyQaxisScan", format="h5")
    if not files: raise FileNotFoundError("No RamseyQaxisScan runs found")
    filepath = files[-1]
    print("(latest RamseyQaxisScan run)")

result = load_h5(filepath)
data, meta, archive = result["data"], result["metadata"], result["archive"]
run_rid = os.path.basename(filepath).split("-")[0]
print("File:", os.path.basename(filepath), "| RID:", run_rid)

# Per-shot arrays; truncate to a common length (guards against a partial
# incremental save where the last sweep is half-written).
shimY   = np.asarray(data["rqs_shimY"], dtype=float)
free_mu = np.asarray(data["rqs_free_time_mu"], dtype=float)
s1 = np.asarray(data["rqs_shot1_spcm1"], float) + np.asarray(data["rqs_shot1_spcm2"], float)
s2 = np.asarray(data["rqs_shot2_spcm1"], float) + np.asarray(data["rqs_shot2_spcm2"], float)
bgt = np.asarray(data["rqs_bg_spcm1"], float) + np.asarray(data["rqs_bg_spcm2"], float)
n = min(len(shimY), len(free_mu), len(s1), len(s2), len(bgt))
shimY, free_mu, shot1_total, shot2_total, bg_total = shimY[:n], free_mu[:n], s1[:n], s2[:n], bgt[:n]

mode = _txt(data.get("rqs_scan_mode"))
fields = np.unique(shimY)
print(f"Mode: {mode} | {len(fields)} B-field(s) | {n} total shots")
print("B-fields (Y-shim, V):", ", ".join(f"{v:.4f}" for v in fields))

## 3. Summary

In [ ]:
print(f"Mode: {mode}   "
      f"detuning_00={_scalar(data.get('snapshot_ramsey_detuning_00_MHz')):.5f} MHz   "
      f"detuning_2ph={_scalar(data.get('snapshot_ramsey_detuning_2ph_MHz')):.5f} MHz   "
      f"step_used={_scalar(data.get('snapshot_ramsey_step_used')):.0f} mu")
print(f"\n{'Y-shim(V)':>10s} {'Nshots':>7s} {'Nft':>5s} {'ft range (ms)':>18s}")
print("-" * 45)
for sy in fields:
    m = (shimY == sy)
    uft = np.unique(free_mu[m])
    print(f"{sy:>10.4f} {int(m.sum()):>7d} {len(uft):>5d} {uft.min()/1e6:>8.2f} .. {uft.max()/1e6:<7.2f}")

## 4. SPCM histogram + manual threshold
Inspect the count distributions, then set `THRESHOLD` at the valley.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(shot1_total, bins=40, color="C2", alpha=0.7, edgecolor="black", lw=0.5)
ax[0].axvline(np.median(shot1_total), color="red", ls="--", label=f"median={np.median(shot1_total):.1f}")
ax[0].set_title("Shot1 (loading)"); ax[0].set_xlabel("SPCM1+SPCM2 counts"); ax[0].legend()
ax[1].hist(shot2_total, bins=40, color="C4", alpha=0.7, edgecolor="black", lw=0.5)
ax[1].axvline(np.median(shot2_total), color="red", ls="--", label=f"median={np.median(shot2_total):.1f}")
ax[1].set_title("Shot2 (post-Ramsey readout)"); ax[1].set_xlabel("SPCM1+SPCM2 counts"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Shot1 median={np.median(shot1_total):.1f}   Shot2 median={np.median(shot2_total):.1f}")

In [ ]:
########################################
THRESHOLD = 13   # set from the histogram valley above
########################################
def retention_vs_freetime(field_sy, threshold):
    """Per-B-field retention vs free-evolution time. Returns (ft_ms, ret, err, n_loaded)."""
    m = (shimY == field_sy)
    ft = free_mu[m]
    loaded = shot1_total[m] > threshold
    retained = shot2_total[m] > threshold
    uft = np.unique(ft)
    ret = np.full(len(uft), np.nan); err = np.full(len(uft), np.nan); nl = np.zeros(len(uft), int)
    for i, t in enumerate(uft):
        mm = (ft == t)
        nlo = int(loaded[mm].sum()); nret = int((loaded[mm] & retained[mm]).sum())
        nl[i] = nlo
        if nlo > 0:
            ret[i] = nret / nlo
            err[i] = np.sqrt(max(ret[i] * (1 - ret[i]), 0.0) / nlo)
    return uft / 1e6, ret, err, nl    # ft converted mu -> ms

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(shot1_total, bins=40, color="C2", alpha=0.5, label="Shot1")
ax.hist(shot2_total, bins=40, color="C4", alpha=0.5, label="Shot2")
ax.axvline(THRESHOLD, color="red", lw=2, ls="--", label=f"Threshold={THRESHOLD}")
ax.set_xlabel("Total SPCM counts"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Threshold={THRESHOLD} | overall loading rate={100 * (shot1_total > THRESHOLD).mean():.1f}%")

## 5. Ramsey LSQ fit per B-field
One fit per B-field (`RamseyFit`: A·cos(2πfx+φ)·exp(-(x/τ)ⁿ)+y0, x in ms). Edge points can be trimmed.

In [ ]:
########################################
EXCLUDE_LEFT = 0
EXCLUDE_RIGHT = 0
########################################
fit_results = {}   # field_sy -> (fitter, result)
nF = len(fields)
ncols = min(nF, 2); nrows = (nF + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), squeeze=False)

print(f"{'Y-shim':>9s} {'freq(kHz)':>11s} {'tau(ms)':>9s} {'n':>5s} {'vis':>6s} {'R2':>6s}")
print("-" * 54)
for k, sy in enumerate(fields):
    ax = axes[k // ncols][k % ncols]
    x_ms, ret, err, nl = retention_vs_freetime(sy, THRESHOLD)
    valid = ~np.isnan(ret)
    vi = np.where(valid)[0]
    if EXCLUDE_LEFT and len(vi) > EXCLUDE_LEFT:
        for idx in vi[:EXCLUDE_LEFT]: valid[idx] = False
    if EXCLUDE_RIGHT and len(vi) > EXCLUDE_RIGHT:
        for idx in vi[-EXCLUDE_RIGHT:]: valid[idx] = False

    ax.errorbar(x_ms[valid], ret[valid], yerr=err[valid], fmt="o", color="C0", ms=4, capsize=2, label="data")
    fitter = RamseyFit(); res = None
    if valid.sum() >= 6:
        xf, yf = x_ms[valid], ret[valid]
        ef = err[valid].copy(); ef[ef == 0] = 0.01
        p0, lo, hi = fitter.initial_guess(xf, yf, yerr=ef)
        res = fitter.fit_lsq(xf, yf, yerr=ef, p0=p0, bounds=(lo, hi))
        if res and res.get("success"):
            xfine = np.linspace(xf.min(), xf.max(), 2000)
            ax.plot(xfine, fitter.evaluate(xfine, res["params"]), "r-", lw=1.2, alpha=0.85, label="LSQ fit")
            f_khz, _ = fitter.frequency_khz(res); tau_ms, _ = fitter.decay_time_ms(res)
            n_exp, _ = fitter.decay_exponent(res); vis, _ = fitter.visibility(res)
            yp = fitter.evaluate(xf, res["params"])
            ss = np.sum((yf - yp) ** 2); st = np.sum((yf - np.mean(yf)) ** 2)
            r2 = 1 - ss / st if st > 0 else 0.0
            print(f"{sy:>9.4f} {f_khz:>11.4f} {tau_ms:>9.2f} {n_exp:>5.2f} {vis:>6.3f} {r2:>6.3f}")
            ax.set_title(f"Y={sy:.4f}V  f={f_khz:.3f}kHz  T2*={tau_ms:.1f}ms")
        else:
            print(f"{sy:>9.4f}  fit FAILED"); ax.set_title(f"Y={sy:.4f}V (fit failed)")
    else:
        print(f"{sy:>9.4f}  too few points ({int(valid.sum())})"); ax.set_title(f"Y={sy:.4f}V (n<6)")
    fit_results[sy] = (fitter, res)
    ax.set_xlabel("Free evolution (ms)"); ax.set_ylabel("Retention")
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=8)
for j in range(nF, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
plt.suptitle(f"{mode} (RID {run_rid}) - Ramsey fit per B-field", y=1.0)
plt.tight_layout(); plt.show()

## 6. Segment zoom per B-field
For each B-field, the 4 free-evolution windows (≈0/8/16/24 ms) plotted separately, with the fit overlaid.

In [ ]:
for sy in fields:
    x_ms, ret, err, nl = retention_vs_freetime(sy, THRESHOLD)
    valid = ~np.isnan(ret)
    xv, rv, ev = x_ms[valid], ret[valid], err[valid]
    if len(xv) < 2:
        print(f"Y={sy:.4f}V: too few points for segment zoom"); continue
    diffs = np.diff(xv); med = np.median(diffs)
    gaps = np.where(diffs > 2 * med)[0]
    starts = np.concatenate([[0], gaps + 1]); ends = np.concatenate([gaps + 1, [len(xv)]])
    nseg = len(starts)
    fitter, res = fit_results.get(sy, (None, None))
    fig, axs = plt.subplots(1, nseg, figsize=(4.2 * nseg, 3.4), sharey=True, squeeze=False)
    axs = axs[0]
    for s, (si, ei) in enumerate(zip(starts, ends)):
        xs, rs, es = xv[si:ei], rv[si:ei], ev[si:ei]
        axs[s].errorbar(xs, rs, yerr=es, fmt="o-", color="C0", ms=4, capsize=2)
        if res and res.get("success") and fitter is not None and len(xs) > 1:
            xf = np.linspace(xs[0], xs[-1], 400)
            axs[s].plot(xf, fitter.evaluate(xf, res["params"]), "r-", lw=1.2, alpha=0.8)
        axs[s].set_ylim(-0.05, 1.05); axs[s].grid(alpha=0.3)
        axs[s].set_title(f"Seg {s+1}: {xs[0]:.2f}-{xs[-1]:.2f} ms"); axs[s].set_xlabel("Free evo (ms)")
    axs[0].set_ylabel("Retention")
    plt.suptitle(f"Y={sy:.4f}V  (RID {run_rid}) - segment zoom", y=1.02)
    plt.tight_layout(); plt.show()

## 7. Coherence vs B-field
Fitted T2* and fringe frequency across the B-field values.

In [ ]:
ys, fs, taus = [], [], []
for sy in fields:
    fitter, res = fit_results.get(sy, (None, None))
    if res and res.get("success"):
        ys.append(sy); fs.append(fitter.frequency_khz(res)[0]); taus.append(fitter.decay_time_ms(res)[0])
if ys:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ys, taus, "o-", color="C0"); ax[0].set_xlabel("Y-shim (V)"); ax[0].set_ylabel("T2* (ms)")
    ax[0].set_title("Coherence time vs B-field"); ax[0].grid(alpha=0.3)
    ax[1].plot(ys, fs, "s-", color="C1"); ax[1].set_xlabel("Y-shim (V)"); ax[1].set_ylabel("Fringe freq (kHz)")
    ax[1].set_title("Fringe frequency vs B-field"); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No successful fits to summarize.")

## 8. Browse datasets

In [ ]:
print("=== datasets/ ===")
for k in sorted(data.keys()):
    v = data[k]
    if isinstance(v, np.ndarray) and v.size > 0:
        print(f"  {k:32s} {str(v.dtype):8s} shape={str(v.shape):12s} [{v.min():.4g} .. {v.max():.4g}]")
    elif isinstance(v, np.ndarray):
        print(f"  {k:32s} {str(v.dtype):8s} (empty)")
    else:
        print(f"  {k:32s} = {_txt(v)}")